In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import json
import warnings

warnings.filterwarnings('ignore')

## incs_no 기준 -> sess_id기준으로 데이터 변경 해야함!

In [2]:
# 추후 로그 나눠서 저장해놓고 쓰는걸로 바꾸기!
log_path = './data/LOG.csv'
msg_path = './data/MSG.csv'
cust_path = './data/CUST.csv'
output_path = './data/grp_sess.csv'

In [3]:
log = pd.read_csv(log_path)
print(log.shape)
log.head(2)

(997147, 100)


,Unnamed: 0,STND_YMD,SITE_ID,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,GOOGLE_CID,SESS_ID,ADID,INCS_NO,...,DVCE_MKT_NM,DVCE_OPRT_NM,DVCE_OPRT_VER,NEW_VISIT_YN,STORE_CD,INVN_ST_YN,SESS_SN_STAY_TIME,EXPS_RSLT_LIST,LIST_TP_NM,ETL_PROC_DTTM
0,0,2024-11-06,UA-110770460-3,WEB,861c6fa067ce9752003b89d3241b51367461c457958400...,2024-11-06 22:22:06.499,2049144370.1730898977,2049144370.17308989771730898977,NaN,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,...,NaN,iOS,iOS 17.6.1,N,NaN,NaN,5.02,NaN,NaN,2024-11-09 09:44:41.372
1,1,2024-11-06,UA-110770460-3,WEB,5f29ad14c1f1785ea41698666b95c2bf469b4bf5122e45...,2024-11-06 22:22:12.875,2049144370.1730898977,2049144370.17308989771730898977,NaN,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,...,NaN,iOS,iOS 17.6.1,N,NaN,NaN,0.60,NaN,NaN,2024-11-09 09:44:41.372


In [43]:
log.columns

Index(['Unnamed: 0', 'STND_YMD', 'SITE_ID', 'WEB_APP_CL_CD', 'LOG_SEQ',
       'LOG_DTTM', 'GOOGLE_CID', 'SESS_ID', 'ADID', 'INCS_NO', 'SITE_MBR_ID',
       'AGE', 'BRTH_YEAR', 'SEX_CD', 'EMP_YN', 'CUST_GRD_NM', 'DVCE_TP_CD',
       'SITE_URL', 'PG_URL', 'PG_NM', 'PG_LOC_VL', 'PG_TP_VL', 'SVC_CL_CD',
       'PG_CNTR_CD', 'PG_LANG_CD', 'UTM_SOURCE', 'UTM_MEDIUM', 'UTM_CAMPAIGN',
       'UTM_CAMPAIGN_ID', 'UTM_CONTENT', 'UTM_TERM', 'ACCM_STAY_TIME_NSS',
       'ACCM_STAY_TIME', 'PG_STAY_TIME', 'SESS_FST_PG_YN', 'SESS_LST_PG_YN',
       'SESS_SN', 'SESS_PV_SN', 'INFL_REFRER', 'SHCT_VST_YN', 'FST_PG_BRA_YN',
       'EVNT_NM', 'EVNT_CAT', 'EVNT_CAT_DTL', 'EVNT_ACTN', 'EVNT_LABEL',
       'PRD_ACTN_TP', 'STLM_STP', 'STLM_STP_OPTN_VL', 'ORD_NO', 'ORD_CPN',
       'CURR_CD', 'ORD_AMT', 'STLM_AMT', 'MIX_STMN_MTHD_CNT', 'STMN_MTHD1',
       'STMN_MTHD2', 'STMN_MTHD3', 'CPN_DC_AMT', 'MBL_GFCR_DC_AMT',
       'BTPN_DC_AMT', 'GIFT_CD_AMT', 'SRCH_WORD', 'SRCH_RSLT_YN', 'SRCH_TP',
       'SRCH_RSLT_C

In [44]:
use_col_list = ['STND_YMD', 'WEB_APP_CL_CD', 'LOG_SEQ', 'LOG_DTTM', 'SESS_ID', 'INCS_NO', 'AGE', 'BRTH_YEAR', 
                'SEX_CD', 'EMP_YN', 'CUST_GRD_NM', 'DVCE_TP_CD', 'SITE_URL', 'PG_URL', 'PG_NM', 'PG_LOC_VL', 
                'PG_TP_VL', 'SVC_CL_CD', 'UTM_SOURCE', 'ACCM_STAY_TIME', 'PG_STAY_TIME', 'EVNT_NM', 'EVNT_CAT_DTL',
                'ORD_AMT', 'CPN_DC_AMT', 'MBL_GFCR_DC_AMT', 'BTPN_DC_AMT', 'GIFT_CD_AMT', 'PRD_INFO', 'DVCE_MDL_NM']

log = log[use_col_list]
log.head(2)

,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,PG_STAY_TIME,EVNT_NM,EVNT_CAT_DTL,ORD_AMT,CPN_DC_AMT,MBL_GFCR_DC_AMT,BTPN_DC_AMT,GIFT_CD_AMT,PRD_INFO,DVCE_MDL_NM
0,2024-11-06,WEB,861c6fa067ce9752003b89d3241b51367461c457958400...,2024-11-06 22:22:06.499,2049144370.17308989771730898977,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,54.0,1970,F,N,...,NaN,login,login,NaN,0.0,0.0,0.0,0.0,NaN,iPhone
1,2024-11-06,WEB,5f29ad14c1f1785ea41698666b95c2bf469b4bf5122e45...,2024-11-06 22:22:12.875,2049144370.17308989771730898977,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,54.0,1970,F,N,...,NaN,user_engagement,NaN,NaN,0.0,0.0,0.0,0.0,NaN,iPhone


### PRD_INFO 파싱

In [45]:
# JSON 형식의 문자열 딕셔너리로 변환하는 함수 정의
def json_str_to_dict(json_str):
    if isinstance(json_str, float) and np.isnan(json_str):
        return {} # 빈 딕셔너리 반환
    pattern = re.compile(r'\"(.*?)\":\s*\"(.*?)\",*\n*')
    matches = pattern.findall(json_str)
    result_dict = {key: (value if value != '(not set)' else None) for key, value in matches}
    return result_dict

# PRD_INFO 컬럼에 함수를 적용하여 파싱된 데이터를 새로운 칼럼으로 추가하는 함수 정의
def apply_and_expand(df, col_name):
    # 각 행의 데이터를 딕셔너리로 변환
    df_dicts = df[col_name].apply(json_str_to_dict)
    
    # 딕셔너리 형태의 데이터를 데이터프레임으로 변환
    expanded_df = pd.json_normalize(df_dicts)
    
    # 생성된 새로운 데이터프레임을 기존 데이터프레임에 병합
    result_df = pd.concat([df, expanded_df], axis=1)
    
    return result_df


# apply_and_expand 함수를 사용하여 데이터프레임을 업데이트
log_prse = apply_and_expand(log, 'PRD_INFO')
print(log_prse.shape)
log_prse.head(2)

(997147, 51)


,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,prd_loc,prd_norm_prc,prd_optn,prd_prom_id,prd_prom_nm,prd_qty,prd_sal_prc,prd_tp_cat_vl,acml_bt_pt,prd_sal_amt
0,2024-11-06,WEB,861c6fa067ce9752003b89d3241b51367461c457958400...,2024-11-06 22:22:06.499,2049144370.17308989771730898977,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,54.0,1970,F,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-06,WEB,5f29ad14c1f1785ea41698666b95c2bf469b4bf5122e45...,2024-11-06 22:22:12.875,2049144370.17308989771730898977,f6eec8e63cee2cf90165c6b3f68c1285d9ffedf7f8e134...,54.0,1970,F,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 로그데이터 그래프 임베딩
* 각 페이지에 상응하는 임베딩 벡터값 필요!

In [46]:
log_prse['PG_EMB_VCT'] = 0

### 로그데이터 확인

In [52]:
grp_sess = pd.DataFrame(log_prse.groupby('SESS_ID'))
grp_sess.columns = ['SESS_ID', 'LOG']
print(grp_sess.shape)
grp_sess.head(2)

(70137, 2)


,SESS_ID,LOG
0,0005ad80698368c16258c365c05c76a21732151843,STND_YMD WEB_APP_CL_CD \ 186383 20...
1,0005ad80698368c16258c365c05c76a21732276772,STND_YMD WEB_APP_CL_CD \ 186389 20...


### 로그 전처리

* 날짜 간격 2일이상 -> 다른 세션으로 분리 / sess_Start를 기준으로 세션 분리
* 포인트 관련 데이터 삭제
* 앱여부 0,1

In [48]:
# 데이터 코딩
len_grp_sess = len(grp_sess)

for i in tqdm(range(len_grp_sess)):
    grp_sess['LOG'][i] = grp_sess['LOG'][i] 
    grp_sess['LOG'][i]['LOG_DTTM'] = pd.to_datetime(grp_sess['LOG'][i]['LOG_DTTM'])
    grp_sess['LOG'][i] = grp_sess['LOG'][i].sort_values(by='LOG_DTTM')
    grp_sess['LOG'][i] = grp_sess['LOG'][i].reset_index(drop=True)
    
    if (grp_sess['LOG'][i]['WEB_APP_CL_CD'] == 'APP').any():
        grp_sess['LOG'][i]['APP'] = 1
    else:
        grp_sess['LOG'][i]['APP'] = 0

100%|██████████| 70137/70137 [32:42<00:00, 35.73it/s]  


In [90]:
temp = grp_sess['LOG'][0]
temp.iloc[-1]['LOG_DTTM']

'2024-11-21 10:19:01.404'

In [91]:
# 데이터 인디케이팅
SESS_TIME_list = []
SRCH_EFRT_list = []
PRDV_CNT_list = []
CAT_CNT_list = []
SAME_PAGE_CNT_list = []
EVNT_CNT_list = []
incs_no = []
lst_sess_time = []

for i in tqdm(range(len_grp_sess)):
    temp = grp_sess['LOG'][i]

    # 전체 세션 시간(time) -> 안쓰는게 나을듯
    SESS_TIME_list.append(np.sum(temp['PG_STAY_TIME']))

    # 탐색 노력(전체 세션 개수)
    SRCH_EFRT_list.append(len(temp))

    # 상품 뷰 숫자
    PRDV_CNT_list.append((len(temp[temp['PRD_INFO'].notna()])))

    # 카테고리 뷰 숫자
    CAT_CNT_list.append(len(set(temp[temp['PRD_INFO'].notna()]['prd_tp_cat_vl'])) - 1)

    # 동일한 페이지를 본 숫자
    SAME_PAGE_CNT_list.append(len(set(temp['PG_URL'])))

    # 이벤트 페이지 탐색 횟수 확인
    EVNT_CNT_list.append(len(temp[temp['EVNT_NM'].notna()]))

    incs_no.append(temp['INCS_NO'])    
    lst_sess_time.append(temp.iloc[-1]['LOG_DTTM'])

100%|██████████| 70137/70137 [04:34<00:00, 255.96it/s]


In [92]:
sess_indicate = pd.DataFrame({'SESS_ID' : grp_sess['SESS_ID'],
        # 'SESS_TIME':SESS_TIME_list,
        'SRCH_EFRT' : SRCH_EFRT_list,
        'PRDV_CNT' : PRDV_CNT_list,
        'CAT_CNT' : CAT_CNT_list,
        'SAMGE_PAGE_CNT' : SAME_PAGE_CNT_list,
        'EVNT_CNT' : EVNT_CNT_list,
        'INCS_NO' : incs_no,
        'LST_SESS_TIME' : lst_sess_time
        })

print(sess_indicate.shape)
sess_indicate.head(2)

(70137, 8)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME
0,0005ad80698368c16258c365c05c76a21732151843,11,4,1,5,11,186383 edda90412258df2570f4e54f2ad5b79b611b...,2024-11-21 10:19:01.404
1,0005ad80698368c16258c365c05c76a21732276772,4,1,0,2,4,186389 edda90412258df2570f4e54f2ad5b79b611b...,2024-11-22 20:59:44.011


### 메시지발송 시간, 재방문 여부에 따른 종속변수 코딩
* sess 이후 메시지 발송 이력이 없으면 -1
* sess 이후 메시지 발송 이력이 있고, 재방문을 했다면 1
* sess 이후 메시지 발송 이력이 있고, 재방문을 하지 않았다면 0
* 메시지 데이터 merge

In [82]:
# 탐색 노력이 8이상인 데이터만 사용 -> 추후 탐색노력에 대한 영향이 없다는 추가 분석 필요
sess_indicate_cutoff = sess_indicate[sess_indicate["SRCH_EFRT"] > 7]

sess_indicate.to_csv('./data/sess_indicate_origin.csv')
sess_indicate_cutoff.to_csv('./data/sess_indicate_cutoff8.csv')